In [1]:
import sys
import platform
import importlib
import importlib.metadata as importlib_metadata
from datetime import datetime

print(f"Python version : {platform.python_version()}")
print(f"Python executable: {sys.executable}")
print(f"Platform       : {platform.platform()}")
print(f"Processor      : {platform.processor() or 'unknown'}")

Python version : 3.11.15
Python executable: /home/iai-labor/anaconda3/envs/traffic/bin/python
Platform       : Linux-7.0.0-30-generic-x86_64-with-glibc2.39
Processor      : x86_64


In [2]:
from importlib.metadata import version, PackageNotFoundError

confirmed_libraries = {
    # package_name (pip/conda name): import_name (if different)
    "duckdb":               None,
    "osmnx":                "osmnx",
    "geopandas":            "geopandas",
    "shapely":              "shapely",
    "holidays":             "holidays",
    "torch":                "torch",
    "pytorch-lightning":    "pytorch_lightning",
    "pytorch-forecasting":  "pytorch_forecasting",
    "pandas":                "pandas",
    "numpy":                 "numpy",
}

def get_version(pkg, import_name=None):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "Not installed"
    except Exception as e:
        return f"Error: {e}"

print(f"{'Package':<22}{'Version':<20}")
print("-" * 42)

for pkg, import_name in confirmed_libraries.items():
    v = get_version(pkg, import_name)
    print(f"{pkg:<22}{v:<20}")

Package               Version             
------------------------------------------
duckdb                1.5.3               
osmnx                 2.1.0               
geopandas             1.1.3               
shapely               2.1.2               
holidays              0.99                
torch                 2.13.0+cu126        
pytorch-lightning     2.2.4               
pytorch-forecasting   1.8.0               
pandas                3.0.3               
numpy                 2.4.6               


In [3]:
likely_libraries = {
    "matplotlib":  "matplotlib",
    "seaborn":     "seaborn",
    "scipy":       "scipy",
    "scikit-learn":"sklearn",
    "pyarrow":     "pyarrow",
    "captum":      "captum",
    "shap":        "shap",
    "tqdm":        "tqdm",
    "requests":    "requests",
}

print(f"{'Package':<22}{'Version':<20}")
print("-" * 42)

for pkg, import_name in likely_libraries.items():
    v = get_version(pkg, import_name)
    print(f"{pkg:<22}{v:<20}")

Package               Version             
------------------------------------------
matplotlib            3.10.9              
seaborn               0.13.2              
scipy                 1.17.1              
scikit-learn          1.9.0               
pyarrow               23.0.1              
captum                0.9.0               
shap                  0.51.0              
tqdm                  4.68.1              
requests              2.34.2              


In [4]:
import ast
import json
from pathlib import Path

PROJECT_DIR = Path("/home/iai-labor/traffic_project")


def extract_imports_from_source(source_code):
    """Parse Python source and return top-level imported package names."""
    found = set()

    try:
        tree = ast.parse(source_code)
    except SyntaxError:
        return found

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                found.add(alias.name.split(".")[0])

        elif isinstance(node, ast.ImportFrom):
            if node.module:
                found.add(node.module.split(".")[0])

    return found


def scan_project(project_dir):
    """Scan .py and .ipynb files for imported packages."""
    import_locations = {}

    if not project_dir.exists():
        print(
            f"Directory not found: {project_dir}\n"
            "Edit PROJECT_DIR above and re-run this cell."
        )
        return set()

    py_files = list(project_dir.rglob("*.py"))
    ipynb_files = list(project_dir.rglob("*.ipynb"))

    for f in py_files:
        try:
            source = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        for pkg in extract_imports_from_source(source):
            import_locations.setdefault(pkg, set()).add(str(f))

    for f in ipynb_files:
        try:
            nb = json.loads(
                f.read_text(encoding="utf-8", errors="ignore")
            )
        except Exception:
            continue

        for cell in nb.get("cells", []):
            if cell.get("cell_type") == "code":
                source = "".join(cell.get("source", []))

                for pkg in extract_imports_from_source(source):
                    import_locations.setdefault(pkg, set()).add(str(f))

    print(
        f"Scanned {len(py_files)} .py files and "
        f"{len(ipynb_files)} .ipynb files under {project_dir}\n"
    )

    return import_locations


# Python standard-library modules to exclude
STDLIB_MODULES = {
    "os", "sys", "re", "json", "types", "pathlib", "collections",
    "itertools", "functools", "datetime", "time", "math", "random",
    "logging", "warnings", "typing", "copy", "ast", "subprocess",
    "argparse", "glob", "shutil", "pickle", "multiprocessing",
    "threading", "io", "csv", "traceback", "platform", "importlib",
    "dataclasses", "enum", "abc", "contextlib", "__future__",
}


import_locations = scan_project(PROJECT_DIR)

third_party = {
    pkg: files
    for pkg, files in import_locations.items()
    if pkg not in STDLIB_MODULES
}


print(f"{'Package (import name)':<25}{'Version':<20}")
print("-" * 45)

for pkg in sorted(third_party):
    v = get_version(pkg, pkg)
    print(f"{pkg:<25}{v:<20}")

Scanned 18 .py files and 61 .ipynb files under /home/iai-labor/traffic_project

Package (import name)    Version             
---------------------------------------------
captum                   0.9.0               
duckdb                   1.5.3               
folium                   0.20.0              
gc                       Not installed       
geopandas                1.1.3               
holidays                 0.99                
joblib                   1.5.3               
lightning                2.6.5               
lime                     0.2.0.1             
matplotlib               3.10.9              
numpy                    2.4.6               
osmnx                    2.1.0               
pandas                   3.0.3               
pkg_resources            Not installed       
pypots                   1.5                 
pytorch_forecasting      1.8.0               
pytorch_lightning        2.2.4               
scipy                    1.17.1              
